<div style="background: #1E293B; border-left: 8px solid #3B82F6; padding: 22px 24px; border-radius: 10px; margin: 15px 0 20px 0;">
<h1 style="margin: 0; color: #FFFFFF; font-size: 28px; font-weight: 700; letter-spacing: -0.5px;">
Molecular Similarity Search Fundamentals
</h1>
<p style="margin: 8px 0 0 0; color: #94A3B8; font-size: 15px;">
RDKit Tutorials | Mathematical Similarity Metrics, Distance Calculations & Chemical Space Screening
</p>
</div>

The **Similar Property Principle** (formulated by Johnson & Maggiora in 1990) asserts that *structurally similar molecules are likely to exhibit similar biological properties*. In modern computer-aided drug design (CADD) and virtual screening, chemical similarity provides a quantitative framework to:
* Identify analogs and bioisosteres of known active lead compounds.
* Screen corporate chemical repositories containing millions of molecules in seconds.
* Cluster and visualize chemical space for diversity selection.
* Featurize compound collections for quantitative structure-activity relationship (QSAR) models.

<div style="background: #1E293B; border-left: 8px solid #3B82F6; padding: 16px 20px; border-radius: 10px; margin: 15px 0 16px 0;">
<h2 style="margin: 0; color: #FFFFFF; font-size: 20px; font-weight: 700;">
1. Mathematical Formulations of Similarity Coefficients
</h2>
</div>

Given two molecular fingerprint bit vectors $A$ and $B$:
* $a$ = Total number of bits set to 1 in molecule $A$
* $b$ = Total number of bits set to 1 in molecule $B$
* $c$ = Number of bits set to 1 in both $A$ and $B$ (bitwise intersection $A \cap B$)

<div style="background: #0F172A; border-left: 5px solid #38BDF8; padding: 12px 18px; border-radius: 8px; margin: 18px 0 12px 0;">
<h3 style="margin: 0; color: #F8FAFC; font-size: 18px; font-weight: 600;">Key Similarity Metrics</h3>
</div>

1. **Tanimoto (Jaccard) Coefficient:**
   $$T_c(A, B) = \frac{c}{a + b - c}$$
   The gold standard in cheminformatics. $T_c \in [0, 1]$. An ECFP4 Tanimoto score $\ge 0.85$ is widely considered a reliable indicator of shared biological activity.

2. **Dice Similarity:**
   $$D(A, B) = \frac{2c}{a + b}$$
   Weights common features more heavily than non-shared features ($D \ge T_c$).

3. **Cosine (Ochiai) Coefficient:**
   $$C(A, B) = \frac{c}{\sqrt{a \times b}}$$
   Geometric mean of the fractions of bits shared relative to each molecule.

4. **Tversky Index (Asymmetric Metric):**
   $$S(A, B; \alpha, \beta) = \frac{c}{\alpha(a - c) + \beta(b - c) + c}$$
   Enables tuning prototype similarity: when $\alpha = 1, \beta = 0$, it measures substructure similarity (is $A$ contained in $B$?).

<div style="background: #1E293B; border-left: 8px solid #3B82F6; padding: 16px 20px; border-radius: 10px; margin: 15px 0 16px 0;">
<h2 style="margin: 0; color: #FFFFFF; font-size: 20px; font-weight: 700;">
2. Pairwise Similarity Calculation in RDKit
</h2>
</div>

Here we generate 2048-bit Morgan fingerprints (radius = 2, equivalent to ECFP4) for a reference collection of non-steroidal anti-inflammatory drugs (NSAIDs) and compute pairwise similarity metrics.

In [ ]:
from rdkit import Chem
from rdkit.Chem import DataStructs
from rdkit.Chem import rdFingerprintGenerator

# Define sample molecules (NSAIDs and related analgesics)
smiles_dict = {
    "Aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "Ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Flurbiprofen": "CC(c1ccc(c(c1)F)c2ccccc2)C(=O)O",
    "Naproxen": "CC(c1ccc2c(c1)cc(cc2)OC)C(=O)O",
    "Paracetamol": "CC(=O)Nc1ccc(O)cc1"
}

# Convert SMILES to RDKit Mol objects
mols = {name: Chem.MolFromSmiles(smi) for name, smi in smiles_dict.items()}

# Generate Morgan fingerprints (ECFP4: radius=2, 2048 bits)
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = {name: mfpgen.GetFingerprint(mol) for name, mol in mols.items()}

# Pairwise comparison between Ibuprofen and Flurbiprofen (structural analogs)
fp_ibu = fps["Ibuprofen"]
fp_flurb = fps["Flurbiprofen"]

tanimoto_sim = DataStructs.TanimotoSimilarity(fp_ibu, fp_flurb)
dice_sim = DataStructs.DiceSimilarity(fp_ibu, fp_flurb)
cosine_sim = DataStructs.CosineSimilarity(fp_ibu, fp_flurb)

print("Pairwise Similarity: Ibuprofen vs Flurbiprofen")
print(f"  * Tanimoto Similarity : {tanimoto_sim:.4f}")
print(f"  * Dice Similarity     : {dice_sim:.4f}")
print(f"  * Cosine Similarity   : {cosine_sim:.4f}")

<div style="background: #1E293B; border-left: 8px solid #3B82F6; padding: 16px 20px; border-radius: 10px; margin: 15px 0 16px 0;">
<h2 style="margin: 0; color: #FFFFFF; font-size: 20px; font-weight: 700;">
3. Bulk Virtual Screening with BulkTanimotoSimilarity
</h2>
</div>

Calculating similarity in a Python loop over large compound databases is inefficient. RDKit provides `DataStructs.BulkTanimotoSimilarity`, which evaluates an entire library in optimized C++ vector instructions.

In [ ]:
# Screen candidate library against Ibuprofen as query
query_name = "Ibuprofen"
query_fp = fps[query_name]

# Prepare target library list excluding the query itself
target_names = [name for name in fps.keys() if name != query_name]
target_fps = [fps[name] for name in target_names]

# Run C++ accelerated bulk screening
bulk_scores = DataStructs.BulkTanimotoSimilarity(query_fp, target_fps)

# Rank candidates by Tanimoto similarity
ranked_candidates = sorted(zip(target_names, bulk_scores), key=lambda x: x[1], reverse=True)

print(f"Virtual Screening Hits for Query Drug: {query_name}")
print("=" * 55)
for rank, (name, score) in enumerate(ranked_candidates, start=1):
    print(f"Rank {rank:2d}: {name:<15} | Tanimoto Score: {score:.4f}")

<div style="background: #1E293B; border-left: 8px solid #3B82F6; padding: 16px 20px; border-radius: 10px; margin: 15px 0 16px 0;">
<h2 style="margin: 0; color: #FFFFFF; font-size: 20px; font-weight: 700;">
4. Summary & Best Practices
</h2>
</div>

* **Default to Morgan / ECFP4:** For similarity searching and ligand-based virtual screening, 2048-bit Morgan fingerprints (radius 2) provide the strongest empirical performance.
* **Tanimoto Significance Threshold:** Scores $\ge 0.85$ indicate high biological likelihood of activity against the same target; scores between $0.50 - 0.70$ often share common scaffold motifs.
* **Vectorized Screening:** Always utilize `DataStructs.BulkTanimotoSimilarity` for screening large compound catalogs ($10^5 - 10^7$ molecules) to benefit from native SIMD vectorization.